# Caffe 模型

In [1]:
import sys
sys.path.extend([".."])
import set_env

ROOT: /media/pc/data/lxw/ai/tvm-book


In [2]:
from tvm_book.tools.frontends import Frontend, TrainInputConfig
from tvm_book.tools import display

caffe 前端模型配置：

In [3]:
print(display.Tree("| ")("models/caffe_demo"))

| caffe_demo
    | config.toml
    | test.caffemodel
    | test.prototxt


{icon}`fa fa-folder-open` `caffe_demo/` 文件夹下存在如下内容：

- {icon}`fa fa-file` `test.caffemodel` 存储 caffe 模型参数的初始化模型
- {icon}`fa fa-file` `test.prototxt` 存储 caffe 模型结构
- {icon}`fa fa-file` `config.toml` 存储 caffe 模型配置信息
    ```{include} models/caffe_demo/config.toml
    :code: toml
    ```

In [5]:
from pathlib import Path
import toml

cfg_path = "models/caffe_demo/config.toml"

cfg_path = Path(cfg_path)
config = toml.load(cfg_path)
frontend = Frontend(config["model"]["model_type"])
if len(config["train_inputs"]) == 1: # "此模型为单输入模型"
    input_config = TrainInputConfig(**config["train_inputs"][0])
    shape_dict = {input_config.name: input_config.shape}
    dtype_dict = {input_config.name: input_config.dtype}

model = frontend.load(
    f"{cfg_path.parent}/{config['model']['init_net_path']}", 
    predict_net_path=f"{cfg_path.parent}/{config['model']['predict_net_path']}", 
    shape_dict=shape_dict, 
    dtype_dict=dtype_dict
)

In [ ]:
import tvm
from tvm import relay

with tvm.transform.PassContext(opt_level=3, disabled_pass={"AlterOpLayout"}):
    mod = relay.quantize.prerequisite_optimize(model.mod, model.params)